# Script 1 — Processamento CVM & EDA
**TCC: Predição de Indicadores Financeiros Corporativos com ML e IA Generativa**

> ⚠️ **Correção aplicada**: arquivos CVM usam `sep=';'` e `encoding='latin1'`

In [ ]:

# ── Dependências ──────────────────────────────────────────────────────────────
import zipfile, os, glob, warnings
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 60)
pd.set_option('display.float_format', '{:,.2f}'.format)

# ── Parâmetros ────────────────────────────────────────────────────────────────
# Coloque os ZIPs da CVM nesta pasta (ex: dfp_cia_aberta_2022.zip, 2023.zip, ...)
PASTA_ZIPS  = Path('dados_cvm')       # ajuste conforme necessário
PASTA_SAIDA = Path('outputs')
PASTA_SAIDA.mkdir(exist_ok=True)

# CORREÇÃO: encoding e separador corretos para arquivos CVM
ENCODING_CVM = 'latin1'
SEP_CVM      = ';'

print("✅ Bibliotecas carregadas")


## 1. Definição das 25 empresas âncora

In [ ]:

# ── 25 Empresas âncora — 5 setores × 5 empresas ──────────────────────────────
EMPRESAS = {
    'Petróleo': {
        'Petrobras':       '33.000.167/0001-01',
        'Prio':            '10.629.105/0001-68',
        'Ultrapar':        '33.256.439/0001-39',
        'Raízen':          '33.453.598/0001-23',
        'Vibra Energia':   '33.453.598/0001-23',  # ajustar CNPJ real
    },
    'Energia': {
        'Engie Brasil':       '02.726.168/0001-97',
        'Equatorial Energia': '02.722.865/0001-82',
        'Taesa':              '07.859.971/0001-30',
        'CPFL Energia':       '02.429.144/0001-93',
        'ISA CTEEP':          '02.998.611/0001-04',
    },
    'Varejo': {
        'Lojas Renner':  '92.754.738/0001-62',
        'Magazine Luiza':'47.960.950/0001-21',
        'Alpargatas':    '61.079.117/0001-05',
        'Arezzo':        '16.590.234/0001-76',
        'Grupo Mateus':  '01.884.051/0001-92',
    },
    'Commodities': {
        'Vale':          '33.592.510/0001-54',
        'Suzano':        '16.404.287/0001-55',
        'Klabin':        '89.637.490/0001-45',
        'Gerdau':        '33.611.500/0001-19',
        'CSN Mineração': '33.042.730/0001-04',
    },
    'Tecnologia': {
        'WEG':       '84.429.695/0001-11',
        'Totvs':     '53.113.791/0001-22',
        'Positivo':  '81.243.735/0001-48',
        'Intelbras': '82.901.000/0001-27',
        'Brisanet':  '24.047.792/0001-60',
    },
}

# Dicionários auxiliares
cnpj_para_nome   = {}
cnpj_para_setor  = {}
nome_para_cnpj   = {}
for setor, emps in EMPRESAS.items():
    for nome, cnpj in emps.items():
        cnpj_para_nome[cnpj]  = nome
        cnpj_para_setor[cnpj] = setor
        nome_para_cnpj[nome]  = cnpj

TODOS_CNPJS = list(cnpj_para_nome.keys())
print(f"Total de empresas: {len(TODOS_CNPJS)}")
for setor, emps in EMPRESAS.items():
    print(f"  {setor}: {list(emps.keys())}")


## 2. Carregamento dos dados CVM

In [ ]:

def normalizar_cnpj(cnpj: str) -> str:
    """Remove pontuação: '33.000.167/0001-01' → '33000167000101'"""
    return ''.join(c for c in cnpj if c.isdigit())

def ler_csv_cvm(zip_path: Path, nome_arquivo: str) -> pd.DataFrame:
    """
    Lê um CSV dentro de um ZIP CVM.
    CORREÇÃO: usa sep=';' e encoding='latin1'
    """
    with zipfile.ZipFile(zip_path) as z:
        nomes = z.namelist()
        matches = [n for n in nomes if nome_arquivo in n]
        if not matches:
            return pd.DataFrame()
        with z.open(matches[0]) as f:
            df = pd.read_csv(
                f,
                sep=ENCODING_CVM and SEP_CVM,   # sep=';'
                encoding=ENCODING_CVM,            # encoding='latin1'
                dtype={'CNPJ_CIA': str, 'CD_CVM': str},
                low_memory=False,
            )
    return df

def carregar_demonstrativo(tipo: str, modalidade: str = 'con') -> pd.DataFrame:
    """
    Carrega todos os ZIPs disponíveis para um tipo de demonstrativo.
    Filtra apenas as 25 empresas âncora.
    tipo: 'DRE', 'BPA', 'BPP', 'DFC_MI', 'DFC_MD', 'DVA', 'DMPL'
    modalidade: 'con' (consolidado) ou 'ind' (individual)
    """
    nome_arquivo = f'dfp_cia_aberta_{tipo}_{modalidade}_'
    zips = sorted(PASTA_ZIPS.glob('dfp_cia_aberta_*.zip'))
    if not zips:
        print(f"⚠️  Nenhum ZIP encontrado em '{PASTA_ZIPS}'. Configure a variável PASTA_ZIPS.")
        return pd.DataFrame()

    partes = []
    for zp in zips:
        df = ler_csv_cvm(zp, nome_arquivo)
        if df.empty:
            continue
        # Normaliza CNPJ para comparação
        df['CNPJ_NORM'] = df['CNPJ_CIA'].apply(normalizar_cnpj)
        cnpjs_norm = [normalizar_cnpj(c) for c in TODOS_CNPJS]
        df = df[df['CNPJ_NORM'].isin(cnpjs_norm)].copy()
        if not df.empty:
            partes.append(df)

    if not partes:
        print(f"⚠️  Nenhum dado encontrado para {tipo}_{modalidade}")
        return pd.DataFrame()

    dfinal = pd.concat(partes, ignore_index=True)

    # Parse de datas — formato já é YYYY-MM-DD
    for col in ['DT_REFER', 'DT_INI_EXERC', 'DT_FIM_EXERC']:
        if col in dfinal.columns:
            dfinal[col] = pd.to_datetime(dfinal[col], errors='coerce')

    # Adiciona metadados
    dfinal['NOME_CIA'] = dfinal['CNPJ_NORM'].map(
        {normalizar_cnpj(k): v for k, v in cnpj_para_nome.items()}
    )
    dfinal['SETOR'] = dfinal['CNPJ_NORM'].map(
        {normalizar_cnpj(k): v for k, v in cnpj_para_setor.items()}
    )
    dfinal['TIPO_DOC'] = tipo
    dfinal['ANO'] = dfinal['DT_REFER'].dt.year

    # Mantém apenas DFP anuais (VERSAO mais alta por empresa/ano)
    if 'VERSAO' in dfinal.columns and 'ORDEM_EXERC' in dfinal.columns:
        dfinal = (dfinal
            .sort_values(['CNPJ_CIA','DT_REFER','VERSAO'], ascending=[True,True,False])
            .drop_duplicates(subset=['CNPJ_CIA','DT_REFER','CD_CONTA','ORDEM_EXERC'])
        )

    print(f"  {tipo}_{modalidade}: {len(dfinal):,} linhas | "
          f"{dfinal['NOME_CIA'].nunique()} empresas | "
          f"anos: {sorted(dfinal['ANO'].dropna().astype(int).unique())}")
    return dfinal

# ── Carrega os demonstrativos ─────────────────────────────────────────────────
print("Carregando demonstrativos (modalidade consolidada)...")
dre  = carregar_demonstrativo('DRE',    'con')
bpa  = carregar_demonstrativo('BPA',    'con')
bpp  = carregar_demonstrativo('BPP',    'con')
dfc  = carregar_demonstrativo('DFC_MI', 'con')
dva  = carregar_demonstrativo('DVA',    'con')
print("\n✅ Carregamento concluído")


## 3. Pivotagem — uma linha por empresa/período/conta

In [ ]:

def pivotar(df: pd.DataFrame, sufixo: str) -> pd.DataFrame:
    """
    Transforma o formato longo (CD_CONTA, VL_CONTA) em formato largo.
    Mantém somente o ÚLTIMO exercício (valor corrente do período).
    """
    if df.empty:
        return pd.DataFrame()

    # Filtra apenas exercício ÚLTIMO (valor do próprio período, não comparativo)
    if 'ORDEM_EXERC' in df.columns:
        df = df[df['ORDEM_EXERC'] == 'ÚLTIMO'].copy()

    # Assegura tipos
    df['VL_CONTA'] = pd.to_numeric(df['VL_CONTA'], errors='coerce')
    df['CD_CONTA'] = df['CD_CONTA'].astype(str).str.strip()

    pivot = df.pivot_table(
        index=['CNPJ_CIA', 'NOME_CIA', 'SETOR', 'ANO', 'DT_REFER'],
        columns='CD_CONTA',
        values='VL_CONTA',
        aggfunc='last',
    )
    pivot.columns = [f'{sufixo}_{c}' for c in pivot.columns]
    return pivot.reset_index()

print("Pivotando demonstrativos...")
p_dre = pivotar(dre, 'DRE')
p_bpa = pivotar(bpa, 'BPA')
p_bpp = pivotar(bpp, 'BPP')
p_dfc = pivotar(dfc, 'DFC')
print(f"  DRE: {p_dre.shape}  BPA: {p_bpa.shape}  BPP: {p_bpp.shape}  DFC: {p_dfc.shape}")

# Merge progressivo
chave = ['CNPJ_CIA', 'NOME_CIA', 'SETOR', 'ANO', 'DT_REFER']
dataset = p_dre.copy() if not p_dre.empty else pd.DataFrame()
for p, nome in [(p_bpa,'BPA'), (p_bpp,'BPP'), (p_dfc,'DFC')]:
    if not p.empty and not dataset.empty:
        dataset = dataset.merge(p, on=chave, how='outer')
    elif not p.empty:
        dataset = p.copy()
    print(f"  Após merge {nome}: {dataset.shape}")

print(f"\n✅ Dataset consolidado: {dataset.shape}")
dataset.head(3)


## 4. Extração de D&A via DFC Método Indireto

In [ ]:

def extrair_dna(dfc_df: pd.DataFrame) -> pd.DataFrame:
    """
    Extrai Depreciação & Amortização das subcontas 6.01.01.xx
    via busca por palavras-chave no campo DS_CONTA.
    """
    if dfc_df.empty:
        return pd.DataFrame()

    PALAVRAS_DNA = ['deprecia', 'amortiza', 'exaust']
    mask = dfc_df['DS_CONTA'].str.lower().str.contains(
        '|'.join(PALAVRAS_DNA), na=False
    )
    dfc_dna = dfc_df[mask].copy()
    dfc_dna['VL_CONTA'] = pd.to_numeric(dfc_dna['VL_CONTA'], errors='coerce').abs()

    dna_por_periodo = (dfc_dna
        .groupby(['CNPJ_CIA', 'NOME_CIA', 'SETOR', 'ANO', 'DT_REFER'])['VL_CONTA']
        .sum()
        .reset_index()
        .rename(columns={'VL_CONTA': 'DNA'})
    )
    print(f"D&A extraído: {len(dna_por_periodo)} registros | "
          f"{dna_por_periodo['NOME_CIA'].nunique()} empresas")
    return dna_por_periodo

dna = extrair_dna(dfc)
if not dna.empty and not dataset.empty:
    dataset = dataset.merge(
        dna[['CNPJ_CIA','ANO','DT_REFER','DNA']],
        on=['CNPJ_CIA','ANO','DT_REFER'], how='left'
    )
    print(f"\nDataset com D&A: {dataset.shape}")
    taxa_dna = dataset['DNA'].notna().mean()
    print(f"  Taxa de cobertura D&A: {taxa_dna:.1%}")


## 5. Cálculo dos 15 KPIs Financeiros

In [ ]:

def calcular_kpis(df: pd.DataFrame) -> pd.DataFrame:
    """Calcula 15 KPIs financeiros a partir dos demonstrativos pivotados."""
    d = df.copy()

    def col(prefixo, conta):
        return f'{prefixo}_{conta}' if f'{prefixo}_{conta}' in d.columns else None

    def get(prefixo, conta):
        c = col(prefixo, conta)
        return d[c] if c else pd.Series(np.nan, index=d.index)

    # Contas principais
    receita    = get('DRE', '3.01')
    lucro_bruto= get('DRE', '3.03')
    ebit       = get('DRE', '3.05')
    lucro_liq  = get('DRE', '3.11')
    desp_fin   = get('DRE', '3.06')
    ativo_tot  = get('BPA', '1')
    ativo_circ = get('BPA', '1.01')
    caixa      = get('BPA', '1.01.01')
    pass_circ  = get('BPP', '2.01')
    div_cp     = get('BPP', '2.01.04')
    div_lp     = get('BPP', '2.02.01')
    pat_liq    = get('BPP', '2.03')
    fco        = get('DFC', '6.01')

    div_bruta  = div_cp.fillna(0) + div_lp.fillna(0)
    div_liq    = div_bruta - caixa.fillna(0)
    dna        = d['DNA'] if 'DNA' in d.columns else pd.Series(0, index=d.index)
    ebitda     = ebit + dna.fillna(0)

    # KPIs
    d['margem_bruta']     = lucro_bruto  / receita.replace(0, np.nan)
    d['margem_ebit']      = ebit         / receita.replace(0, np.nan)
    d['margem_liquida']   = lucro_liq    / receita.replace(0, np.nan)
    d['margem_ebitda']    = ebitda       / receita.replace(0, np.nan)
    d['roe']              = lucro_liq    / pat_liq.replace(0, np.nan)
    d['roa']              = lucro_liq    / ativo_tot.replace(0, np.nan)
    d['liquidez_corrente']= ativo_circ   / pass_circ.replace(0, np.nan)
    d['liquidez_imediata']= caixa        / pass_circ.replace(0, np.nan)
    d['endividamento']    = (div_bruta)  / ativo_tot.replace(0, np.nan)
    d['alavancagem_de']   = div_bruta    / pat_liq.replace(0, np.nan)
    d['div_liquida']      = div_liq
    d['cobertura_juros']  = ebit         / desp_fin.abs().replace(0, np.nan)
    d['giro_ativo']       = receita      / ativo_tot.replace(0, np.nan)
    d['fco_receita']      = fco          / receita.replace(0, np.nan)
    d['fco_lucro']        = fco          / lucro_liq.replace(0, np.nan)
    d['EBITDA']           = ebitda

    KPIS = ['margem_bruta','margem_ebit','margem_liquida','margem_ebitda',
            'roe','roa','liquidez_corrente','liquidez_imediata',
            'endividamento','alavancagem_de','div_liquida','cobertura_juros',
            'giro_ativo','fco_receita','fco_lucro','EBITDA']

    cobertura = d[KPIS].notna().mean().sort_values()
    print("Cobertura dos KPIs:")
    for kpi, val in cobertura.items():
        print(f"  {kpi:<25} {val:.0%}")
    return d

if not dataset.empty:
    dataset = calcular_kpis(dataset)
    print(f"\nDataset com KPIs: {dataset.shape}")


## 6–14. Análise Exploratória de Dados (9 Blocos)

In [ ]:

if dataset.empty:
    print("⚠️  Dataset vazio — execute as células anteriores com os ZIPs CVM.")
else:
    # ── BLOCO 1: Visão geral ─────────────────────────────────────────────────
    print("=" * 60)
    print("BLOCO 1 — Visão Geral do Dataset")
    print("=" * 60)
    print(f"  Linhas: {len(dataset):,}  |  Colunas: {dataset.shape[1]}")
    print(f"  Empresas: {dataset['NOME_CIA'].nunique()}")
    print(f"  Setores: {dataset['SETOR'].nunique()}")
    print(f"  Anos: {sorted(dataset['ANO'].dropna().astype(int).unique())}")
    print(f"  Nulos global: {dataset.isnull().mean().mean():.1%}")


In [ ]:

if not dataset.empty:
    # ── BLOCO 2: Cobertura temporal ──────────────────────────────────────────
    print("BLOCO 2 — Cobertura Temporal por Empresa")
    cobertura_temp = (dataset.groupby('NOME_CIA')['ANO']
                      .agg(['min','max','count'])
                      .rename(columns={'min':'Primeiro','max':'Último','count':'Registros'})
                      .sort_values('Registros', ascending=False))
    print(cobertura_temp.to_string())


In [ ]:

if not dataset.empty:
    # ── BLOCO 3: Inventário de contas ────────────────────────────────────────
    print("BLOCO 3 — Inventário de Contas Disponíveis")
    for prefixo in ['DRE', 'BPA', 'BPP', 'DFC']:
        cols = [c for c in dataset.columns if c.startswith(f'{prefixo}_')]
        cobertura = dataset[cols].notna().mean().sort_values(ascending=False)
        print(f"  {prefixo}: {len(cols)} contas")
        print(f"    Top 5: {list(cobertura.head(5).index)}")


In [ ]:

if not dataset.empty:
    # ── BLOCO 4: Qualidade dos dados ─────────────────────────────────────────
    print("BLOCO 4 — Qualidade dos Dados")
    KPIS = ['margem_bruta','margem_ebit','margem_liquida','margem_ebitda',
            'roe','roa','liquidez_corrente','liquidez_imediata',
            'endividamento','alavancagem_de','div_liquida','cobertura_juros',
            'giro_ativo','fco_receita','fco_lucro','EBITDA']
    nulos = dataset[KPIS].isnull().mean().sort_values(ascending=False)
    print("  Taxas de nulos por KPI:")
    print(nulos.to_string())

    # Outliers (z-score > 5)
    z = np.abs(stats.zscore(dataset[KPIS].dropna(), axis=0))
    outliers = (z > 5).sum()
    print(f"\n  Outliers extremos (|z| > 5): {outliers.sum()}")


In [ ]:

if not dataset.empty:
    # ── BLOCO 5: Estatísticas descritivas ────────────────────────────────────
    print("BLOCO 5 — Estatísticas Descritivas dos KPIs")
    stats_kpis = dataset[KPIS].describe().T
    print(stats_kpis[['mean','std','min','50%','max']].round(3).to_string())


In [ ]:

if not dataset.empty:
    # ── BLOCO 6: Targets principais ──────────────────────────────────────────
    print("BLOCO 6 — Targets Principais")
    targets_cols = [c for c in dataset.columns if c in
                    ['DRE_3.01','DRE_3.11','EBITDA']]
    if targets_cols:
        print(dataset.groupby('SETOR')[targets_cols].median().round(0).to_string())


In [ ]:

if not dataset.empty:
    # ── BLOCO 7: Correlações com os targets ──────────────────────────────────
    print("BLOCO 7 — Correlações (Pearson) com Target Receita Líquida")
    if 'DRE_3.01' in dataset.columns:
        corr = dataset[KPIS + ['DRE_3.01']].corr()['DRE_3.01'].drop('DRE_3.01')
        print(corr.sort_values(key=abs, ascending=False).head(10).to_string())


In [ ]:

if not dataset.empty:
    # ── BLOCO 8: Evolução temporal por setor ─────────────────────────────────
    print("BLOCO 8 — Evolução Temporal de Receita por Setor")
    if 'DRE_3.01' in dataset.columns:
        evol = dataset.groupby(['SETOR','ANO'])['DRE_3.01'].median().unstack('SETOR')
        print(evol.round(0).to_string())

        fig, ax = plt.subplots(figsize=(12, 5))
        evol.plot(ax=ax, marker='o')
        ax.set_title('Receita Líquida Mediana por Setor (R$ mil)')
        ax.set_xlabel('Ano'); ax.set_ylabel('R$ mil')
        ax.legend(loc='upper left')
        plt.tight_layout()
        plt.savefig(PASTA_SAIDA / 'evol_receita_setor.png', dpi=150)
        plt.show()


In [ ]:

if not dataset.empty:
    # ── BLOCO 9: Inventário final de KPIs disponíveis ────────────────────────
    print("BLOCO 9 — Inventário Final de KPIs para Modelagem")
    kpis_disponiveis = [k for k in KPIS if dataset[k].notna().mean() > 0.5]
    print(f"  KPIs com >50% de cobertura: {len(kpis_disponiveis)}")
    for k in kpis_disponiveis:
        print(f"    ✅ {k}: {dataset[k].notna().mean():.0%}")
    kpis_excluidos = [k for k in KPIS if k not in kpis_disponiveis]
    for k in kpis_excluidos:
        print(f"    ❌ {k}: {dataset[k].notna().mean():.0%} (excluído)")


## 7. Salvar dataset consolidado

In [ ]:

if not dataset.empty:
    caminho = PASTA_SAIDA / 'dataset_cvm_consolidado.parquet'
    dataset.to_parquet(caminho, index=False)
    print(f"✅ Dataset salvo em: {caminho}")
    print(f"   Shape: {dataset.shape}")
    print(f"   Colunas principais: {list(dataset.columns[:20])}")
